## Cell 1: Load labeled data 

In [1]:
import pandas as pd
import numpy as np

df = pd.read_pickle('C:/Users/santh/battery-rul-project/data/processed/labeled_battery_data.pkl')
df.head()

,battery,cycle_idx,type,voltage,current,temperature,capacity_Ah,initial_capacity,SOH,RUL
0,B0005,1,discharge,"[4.191491807505295, 4.191004927950373, 4.12360...","[-0.004901589207462691, -0.002657367145453474,...","[24.330033885570543, 24.327385288702907, 24.34...",1.856487,1.856487,1.000000,589
1,B0005,3,discharge,"[4.189773213846608, 4.1891915832591025, 4.1250...","[2.125117981080765e-05, -0.0005661765024619325...","[24.697751935729325, 24.69005382346846, 24.701...",1.846327,1.856487,0.994527,587
2,B0005,5,discharge,"[4.188186735991303, 4.187545434539934, 4.12552...","[-0.0017540301662326099, -0.001778437560872837...","[24.734266163954402, 24.738310956683897, 24.75...",1.835349,1.856487,0.988614,585
3,B0005,7,discharge,"[4.188461118855572, 4.188003528970895, 4.12777...","[-0.0027750361294468047, -0.000832934037246042...","[24.65423646922845, 24.652950193336693, 24.665...",1.835263,1.856487,0.988567,583
4,B0005,9,discharge,"[4.188298524761055, 4.187708684562886, 4.12769...","[-0.007980866803888688, -0.0017310739133212117...","[24.524796959348127, 24.52086174427053, 24.534...",1.834646,1.856487,0.988235,581


## Cell 2: Extracting scalar features from each cycle (voltage/current/temp arrays)

In [2]:
def extract_features(row):
    v = np.array(row['voltage'])
    i = np.array(row['current'])
    t = np.array(row['temperature'])
    x = np.arange(len(v))

    # Voltage features
    v_mean, v_std, v_min, v_max = v.mean(), v.std(), v.min(), v.max()
    v_slope = np.polyfit(x, v, 1)[0]          # how fast/slow the discharge curve drops
    v_range = v_max - v_min

    # Current features
    i_mean, i_std = i.mean(), i.std()

    # Temperature features
    t_mean, t_max, t_min = t.mean(), t.max(), t.min()
    t_rise = t_max - t_min                     # how much heat rise occurred during discharge

    return pd.Series({
        'v_mean': v_mean, 'v_std': v_std, 'v_min': v_min, 'v_max': v_max,
        'v_slope': v_slope, 'v_range': v_range,
        'i_mean': i_mean, 'i_std': i_std,
        't_mean': t_mean, 't_max': t_max, 't_min': t_min, 't_rise': t_rise,
    })

feat_df = df.apply(extract_features, axis=1)
feat_df = pd.concat([df[['battery', 'cycle_idx', 'capacity_Ah', 'SOH', 'RUL']], feat_df], axis=1)
feat_df.head()

,battery,cycle_idx,capacity_Ah,SOH,RUL,v_mean,v_std,v_min,v_max,v_slope,v_range,i_mean,i_std,t_mean,t_max,t_min,t_rise
0,B0005,1,1.856487,1.000000,589,3.529343,0.234954,2.618762,4.191492,-0.002526,1.572729,-1.822711,0.582457,32.578022,38.951894,24.327385,14.624508
1,B0005,3,1.846327,0.994527,587,3.536819,0.233775,2.604616,4.189773,-0.002502,1.585157,-1.821613,0.583789,32.730829,39.003326,24.690054,14.313272
2,B0005,5,1.835349,0.988614,585,3.543293,0.226301,2.700208,4.188187,-0.002433,1.487979,-1.820274,0.585058,32.648494,38.802209,24.734266,14.067943
3,B0005,7,1.835263,0.988567,583,3.543247,0.231448,2.617717,4.188461,-0.002479,1.570744,-1.829303,0.572175,32.520265,38.725760,24.652950,14.072810
4,B0005,9,1.834646,0.988235,581,3.541929,0.235356,2.574526,4.188299,-0.002512,1.613772,-1.829833,0.572161,32.387717,38.630248,24.520862,14.109386


## Cell 3: Adding cycle-to-cycle delta features (to capture the degradation trend

In [3]:
feat_df = feat_df.sort_values(['battery', 'cycle_idx']).reset_index(drop=True)

feat_df['capacity_delta'] = feat_df.groupby('battery')['capacity_Ah'].diff().fillna(0)
feat_df['v_mean_delta'] = feat_df.groupby('battery')['v_mean'].diff().fillna(0)
feat_df['t_rise_delta'] = feat_df.groupby('battery')['t_rise'].diff().fillna(0)

feat_df.head()

,battery,cycle_idx,capacity_Ah,SOH,RUL,v_mean,v_std,v_min,v_max,v_slope,v_range,i_mean,i_std,t_mean,t_max,t_min,t_rise,capacity_delta,v_mean_delta,t_rise_delta
0,B0005,1,1.856487,1.000000,589,3.529343,0.234954,2.618762,4.191492,-0.002526,1.572729,-1.822711,0.582457,32.578022,38.951894,24.327385,14.624508,0.000000,0.000000,0.000000
1,B0005,3,1.846327,0.994527,587,3.536819,0.233775,2.604616,4.189773,-0.002502,1.585157,-1.821613,0.583789,32.730829,39.003326,24.690054,14.313272,-0.010160,0.007476,-0.311236
2,B0005,5,1.835349,0.988614,585,3.543293,0.226301,2.700208,4.188187,-0.002433,1.487979,-1.820274,0.585058,32.648494,38.802209,24.734266,14.067943,-0.010978,0.006474,-0.245329
3,B0005,7,1.835263,0.988567,583,3.543247,0.231448,2.617717,4.188461,-0.002479,1.570744,-1.829303,0.572175,32.520265,38.725760,24.652950,14.072810,-0.000087,-0.000046,0.004867
4,B0005,9,1.834646,0.988235,581,3.541929,0.235356,2.574526,4.188299,-0.002512,1.613772,-1.829833,0.572161,32.387717,38.630248,24.520862,14.109386,-0.000617,-0.001319,0.036576


## Cell 4: sanity check (missing values, ranges)

In [4]:
print(feat_df.isnull().sum())
print(feat_df.describe())

battery           0
cycle_idx         0
capacity_Ah       0
SOH               0
RUL               0
v_mean            0
v_std             0
v_min             0
v_max             0
v_slope           0
v_range           0
i_mean            0
i_std             0
t_mean            0
t_max             0
t_min             0
t_rise            0
capacity_delta    0
v_mean_delta      0
t_rise_delta      0
dtype: int64
        cycle_idx  capacity_Ah         SOH         RUL      v_mean  \
count  636.000000   636.000000  636.000000  636.000000  636.000000   
mean   265.426101     1.581652    0.828482  226.182390    3.498637   
std    177.088365     0.198765    0.109306  178.153773    0.047579   
min      1.000000     1.153818    0.566893    0.000000    3.403906   
25%    115.250000     1.421123    0.748067   70.750000    3.466448   
50%    246.000000     1.559695    0.823178  202.500000    3.502593   
75%    410.000000     1.763486    0.925262  343.250000    3.540966   
max    612.000000     2.035

## Cell 5: save feature dataset

In [5]:
feat_df.to_pickle('C:/Users/santh/battery-rul-project/data/processed/feature_battery_data.pkl')
print("Saved feature dataset:", feat_df.shape)

Saved feature dataset: (636, 20)
